In [1]:
import requests
# Function used to search for model IDs for various LLMs.
# Model ID is required for API calls.
response = requests.get("https://openrouter.ai/api/v1/models")

models = response.json()["data"]

# naturally, deepseek models are used as those are cheap.
search_term = "deepseek"

for model in models:
    if search_term.lower() in model["name"].lower():
        print(f"Name: {model['name']}")
        print(f"ID:   {model['id']}")
        print("----------------------")

Name: DeepSeek: DeepSeek V4.1 Flash 1017
ID:   deepseek/deepseek-v4.1-flash-1017
----------------------
Name: DeepSeek: DeepSeek V4.1 Flash 1017 (batch)
ID:   deepseek/deepseek-v4.1-flash-1017:batch
----------------------
Name: DeepSeek: DeepSeek Pro Latest
ID:   ~deepseek/deepseek-pro-latest
----------------------
Name: DeepSeek: DeepSeek Flash Latest
ID:   ~deepseek/deepseek-flash-latest
----------------------
Name: DeepSeek: DeepSeek V4.1 Flash
ID:   deepseek/deepseek-v4.1-flash
----------------------
Name: DeepSeek: DeepSeek V4.1 Flash (batch)
ID:   deepseek/deepseek-v4.1-flash:batch
----------------------
Name: DeepSeek: DeepSeek V4 Flash Vision Exp
ID:   deepseek/deepseek-v4-flash-vision-exp
----------------------
Name: DeepSeek: DeepSeek V4 Pro 0813
ID:   deepseek/deepseek-v4-pro-0813
----------------------
Name: DeepSeek: DeepSeek V4 Flash Latest
ID:   ~deepseek/deepseek-v4-flash-latest
----------------------
Name: DeepSeek: DeepSeek V4 Flash 0731
ID:   deepseek/deepseek-v4-fla

In [ ]:
!pip install openai

In [3]:
# create connection to Open router
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("OPENROUTER_API_KEY")
client = OpenAI(base_url="https://openrouter.ai/api/v1",api_key=api_key)

In [4]:
MODEL = "deepseek/deepseek-v4.1-flash-1017"

TEMPERATURE = 0.7

MAX_TOKENS = 1000

# prompts are generated with AI; AI is better at prompting AI!
INTERVENTION_PROMPTS = {
    1: """
    Gently check in with the test taker.
    Acknowledge their frustration without being patronizing.
    Offer support, but do not give hints or solve the problem.
    Keep your response brief.
    """,

    2: """
    Offer the test taker a small hint.
    Help them think about the problem without giving away the answer.
    Keep the hint relevant to the current problem.
    """,

    3: """
    Provide guided help.
    Break the problem into smaller, manageable steps.
    Ask questions that help the test taker reason through the problem.
    Avoid simply giving away the answer.
    """
}
## Conversation history is stored as [{role:"",content:""}]
## This enables a helpful chat, not just a helpful response. Each API request requires the entire chat
## to be passed as context, as it does not store history server side.
conversation = [
    {
        "role": "user",
        "content": "I don't understand this question."
    },
    {
        "role": "assistant",
        "content": "That's okay. Which part is confusing?"
    },
    {
        "role": "user",
        "content": "I don't understand the second sentence."
    }
]






In [ ]:
# Responds to tuple output by NN
# Builds conversational history,
# And gets the response from the LLM in question.
def generate_response(input_tuple, user_message, conversation):

    # Extract intervention level
    level = input_tuple[1]

    # Remember the test taker's message
    # Staying quiet does not mean conversation history should be ignored.
    conversation.append({"role": "user","content": user_message})

    # Level 0: Stay quiet
    if level == 0:
        return None

    # This sets the tone of the response; largely helps keep model on topic.
    # Assisting prompts generated by LLMs which are not from the same company seem to work better,
    # by allowing a wider "voice" for the agent.
    system_prompt = """
    You are an intervention assistant supporting a test taker
    who may be experiencing frustration.

    Your goal is to provide appropriate support without
    unnecessarily interrupting the test taker's work.

    Adapt your behaviour according to the intervention instructions.
    """

    system_prompt += INTERVENTION_PROMPTS[level]

    # Add to conversation history
    messages = [{"role": "system","content": system_prompt}] + conversation

    response = client.chat.completions.create(model=MODEL,temperature=TEMPERATURE,max_tokens=MAX_TOKENS,messages=messages)

    assistant_message = response.choices[0].message.content

    # Save response in conversation history
    conversation.append({"role": "assistant","content": assistant_message})

    return assistant_message

In [ ]:
# Conversations are saved as CSVs for analysis later.
import csv
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
CSV_FILENAME = f"llm_test_session_{timestamp}.csv"

# Create CSV file and write headers
with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)

    writer.writerow([
        "Turn",
        "Timestamp",
        "User_Message",
        "Intervention_Level",
        "Intervention_Prompt",
        "Assistant_Response",
        "Intervention_Provided"
    ])

print(f"Session will be saved to: {CSV_FILENAME}")

Session will be saved to: llm_test_session_20261115_180924.csv


In [ ]:
conversation = []

turn_number = 0

print("========== INTERACTIVE LLM TESTING ==========")
print("Type 'exit' to end the conversation.")
print("---------------------------------------------")

while True:

    user_message = input("\nTEST TAKER: ")

    if user_message.lower() in ["exit", "quit"]:
        print("Ending conversation.")
        break

    # Validate intervention level
    while True:
        try:
            level = int(input("Intervention level (0-3): "))
            if level in [0, 1, 2, 3]:
                break
            print("Please enter a number between 0 and 3.")
        except ValueError:
            print("Please enter a valid number.")

    turn_number += 1

    # Generate assistant response
    result = generate_response(
        (True, level),
        user_message,
        conversation
    )

    # Identify intervention prompt
    intervention_prompt = (
        INTERVENTION_PROMPTS[level]
        if level != 0
        else "No intervention"
    )

    # Display response
    if result is not None:
        print("\nASSISTANT:", result)
    else:
        print("\nASSISTANT: [No intervention]")

    # Save turn to CSV
    with open(CSV_FILENAME, mode="a", newline="", encoding="utf-8") as file:

        writer = csv.writer(file)

        writer.writerow([
            turn_number,
            datetime.now().isoformat(),
            user_message,
            level,
            intervention_prompt,
            result if result is not None else "",
            level != 0
        ])

    print("\n[Turn saved to CSV]")

========== INTERACTIVE LLM TESTING ==========
Type 'exit' to end the conversation.
---------------------------------------------

TEST TAKER:  How do I use if–then statements to control the flow of a program?
Intervention level (0-3): 3

ASSISTANT: Let’s break it into small steps.

1. **Name the decision your program needs to make.**  
   Ask: *“What condition am I checking?”*  
   Example idea: “Is the score high enough?” or “Is the password correct?”

2. **Write the condition as something that can be true or false.**  
   Ask: *“What values or variables does this condition depend on?”*

3. **Decide what should happen when the condition is true.**  
   Ask: *“Which action belongs inside the `then` branch?”*

4. **Decide what should happen when it is false — if anything.**  
   Ask: *“Do I need an `else` branch, or should the program just continue?”*

5. **Trace it with a sample value.**  
   Ask: *“If the condition were true right now, which line would run? If false, which line would 

KeyboardInterrupt: Interrupted by user

In [ ]:
# Run this cell to save chat history
# Resulting files may have null characters due to OS language encodings,
# or the model using UTF characters which simply cannot be displayed as plain text.
from google.colab import files

files.download(CSV_FILENAME)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>